## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-02-09-55-26 +0000,nypost,Susan Collins accuses Schumer of pushing attac...,https://nypost.com/2026/11/02/us-news/susan-co...
1,2026-11-02-09-52-23 +0000,nyt,Binance Reports $3.64 Billion in Outflows as I...,https://www.nytimes.com/2026/11/01/business/bi...
2,2026-11-02-09-46-46 +0000,wsj,Stock Market Today: Stock Futures Recover Afte...,https://www.wsj.com/livecoverage/stock-market-...
3,2026-11-02-09-46-00 +0000,wsj,Dollar Falls to 1-Month Low as Oil Prices Slip,https://www.wsj.com/finance/currencies/asian-c...
4,2026-11-02-09-39-00 +0000,wsj,"Stock Futures Higher, Oil Lower as Investors R...",https://www.wsj.com/finance/stocks/stock-futur...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2656/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
184,trump,20
12,race,17
166,senate,16
188,election,12
172,michigan,12


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
44,2026-11-02-05-08-00 +0000,wsj,Rep. Haley Stevens stunned Democrats on Sunday...,https://www.wsj.com/politics/elections/how-rep...,137
151,2026-11-01-18-25-38 +0000,chitri,Democrat Haley Stevens endorses GOP’s Mike Rog...,https://www.chicagotribune.com/2026/11/01/demo...,104
104,2026-11-01-22-44-56 +0000,nypost,Michigan Senate candidate Mike Rogers says he’...,https://nypost.com/2026/11/01/us-news/michigan...,82
161,2026-11-01-17-37-01 +0000,nypost,Abdul El-Sayed’s ex-Dem opponent Rep. Haley St...,https://nypost.com/2026/11/01/us-news/abdul-el...,81
106,2026-11-01-22-35-48 +0000,nyt,Haley Stevens Bashed Mike Rogers in Michigan S...,https://www.nytimes.com/2026/11/01/us/election...,78


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
44,137,2026-11-02-05-08-00 +0000,wsj,Rep. Haley Stevens stunned Democrats on Sunday...,https://www.wsj.com/politics/elections/how-rep...
53,63,2026-11-02-04-04-28 +0000,bbc,Trump makes final campaign push in final weeke...,https://www.bbc.co.uk/news/articles/c3j47gjx29...
134,47,2026-11-01-20-00-00 +0000,wsj,"In the Florida governor’s race, Democrat David...",https://www.wsj.com/politics/elections/democra...
133,38,2026-11-01-20-02-00 +0000,wsj,Maybe the Democrats win big Tuesday. But the p...,https://www.wsj.com/politics/policy/democrats-...
31,35,2026-11-02-07-10-00 +0000,wsj,Treasury Yields Little Changed in Asian Trade ...,https://www.wsj.com/finance/investing/treasury...
89,33,2026-11-02-00-00-00 +0000,wsj,Miami’s luxurious Design District is designate...,https://www.wsj.com/politics/policy/miami-oppo...
130,33,2026-11-01-20-21-30 +0000,nypost,Alaska Sen. Sullivan warns alleged ‘Decoy Dan’...,https://nypost.com/2026/11/01/us-news/alaska-s...
24,32,2026-11-02-07-45-39 +0000,nypost,Crazed University of Michigan student pushes c...,https://nypost.com/2026/11/02/us-news/universi...
195,32,2026-11-01-15-15-00 +0000,wsj,Voters head to the polls on Tuesday as benefic...,https://www.wsj.com/economy/us-economic-data-g...
32,26,2026-11-02-07-05-05 +0000,startribune,Jet nosedives and erupts into fireball near bu...,https://www.startribune.com/business-jet-carry...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
